<a href="https://colab.research.google.com/github/junseo25/ds2002-walmart-hurricane-project/blob/main/Weather_cleaning5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##copy past from previous labs and studios | From 2026-10-07 — APIs 101 and Ingestion with Retries — Studio
#scratch work - irrelevant

In [1]:
import requests, pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta


BASE = 'https://archive-api.open-meteo.com/v1/archive'

END = date.today() - timedelta(days=3)     # let the archive settle
START = END - timedelta(days=2)            # a three-day window

params = {
    'latitude': 38.03, 'longitude': -78.51,          # Charlottesville
    'start_date': START.isoformat(),
    'end_date': END.isoformat(),
    'hourly': 'temperature_2m,precipitation',
    'timezone': 'America/New_York',
}

print(BASE + '?' + urlencode(params))



resp = requests.get(BASE, params=params, timeout=15)

print('status:', resp.status_code)

if resp.status_code == 200:
    data = resp.json()
    print('keys:', list(data.keys()))
    print('units:', data['hourly_units'])

    hourly = pd.DataFrame(data['hourly'])
    hourly['time'] = pd.to_datetime(hourly['time'])
    print(hourly.shape)
    print(hourly.head())

    wettest = hourly.loc[hourly['precipitation'].idxmax()]
    print('wettest hour:', wettest['time'], '|', wettest['precipitation'], 'mm')
else:
    print('Failed to retrieve data. Error details:', resp.json())

https://archive-api.open-meteo.com/v1/archive?latitude=38.03&longitude=-78.51&start_date=2026-11-01&end_date=2026-11-03&hourly=temperature_2m%2Cprecipitation&timezone=America%2FNew_York
status: 200
keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
units: {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm'}
(72, 3)
                 time  temperature_2m  precipitation
0 2026-11-01 00:00:00             9.7            0.0
1 2026-11-01 01:00:00             7.8            0.0
2 2026-11-01 02:00:00             8.6            0.0
3 2026-11-01 03:00:00             9.5            0.0
4 2026-11-01 04:00:00             9.8            0.0
wettest hour: 2026-11-02 08:00:00 | 0.1 mm


##Need this to be editable and malleable for other time zones and lat long coordinates based on data file
#Scratch work, irrelevant



In [2]:
from pandas.io.api import read_csv
# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


#Scratch work - irrelevant


In [ ]:
import requests
import pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta

#Since every store is in the same state, just set all state rows to the same value.
store['state'] = 'FL'

#Tallahassee is missing coords instead of dropping a store location
store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]

#drop any duplicate rows that may have the same latitude and longitude
store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)

#copied from lab
END = date.today() - timedelta(days=3)
START = END - timedelta(days=2)

BASE = 'https://archive-api.open-meteo.com/v1/archive'

#generates an empty list to inpute data into outside of the for loop so it doesnt get reset
all_weather_data = []
#iterates through all the rows so that the parameters are all correct
for i in range(len(store)):
    store_id = store.loc[i, 'store_id']
    city = store.loc[i, 'city']
    lat = store.loc[i, 'latitude']
    lon = store.loc[i, 'longitude']
    state = store.loc[i, 'state']

    # Assign appropriate timezone (Pensacola is in the Central Time Zone)
    tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'

#parameters that the API looks for, fixed by the if else statement above for tz
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': START.isoformat(),
        'end_date': END.isoformat(),
        'hourly': 'temperature_2m,precipitation',
        'timezone': tz,
    }

    response = requests.get(BASE, params=params)

#The response code is a successful one, no error occurs
    if response.status_code == 200:
        data = response.json()
        #Prints our that it can display the data for a specific store
        print(f"Successfully fetched data for {store_id} ({city}, {state})")
#finds hourly data for each store, if hourly data at a specific hour does not exist, it returns empty value
        hourly = data.get('hourly', {})
        print(hourly)
        if hourly:
            temp_stores = pd.DataFrame({
                'store_id': store_id,
                'city': city,
                'state': state,
                'time': hourly.get('time'),
                'temperature_2m': hourly.get('temperature_2m'),
                'precipitation': hourly.get('precipitation')
            })
#adds the data to the list
            all_weather_data.append(temp_stores)
    else:
        print(f"Failed to fetch data for {store_id} ({city}): Status {response.status_code}")





In [ ]:
if data:
    hourly = pd.DataFrame(data['hourly'])
else:                                  # keeps the notebook usable offline
    hourly = pd.DataFrame({
        'time': pd.date_range(START, periods=72, freq='h').astype(str),
        'temperature_2m': [12 + (i % 12) for i in range(72)],
        'precipitation': [0.0] * 30 + [1.4] * 6 + [0.0] * 36,
    })

hourly['time'] = pd.to_datetime(hourly['time'])

daily = (hourly.groupby(hourly['time'].dt.date)
         .agg(precipitation_total=('precipitation', 'sum'),
              temp_max=('temperature_2m', 'max'))
         .reset_index()
         .rename(columns={'time': 'date'}))
daily



# Combine into a master DataFrame
print(all_weather_data)
if all_weather_data:
    master_weather_df = pd.concat(all_weather_data, ignore_index=True)
    print(master_weather_df.head())

#PART 2: MAKING THE CODE WITH FUNCTIONS AFTER THE TESTING ABOVE
#Three functions
1. Cleaning store dataframe
2. Generating weather data for a single store
3. Looping through function 2 to generate comprehensive data for all stores

In [17]:
import requests, pandas as pd, numpy as np, requests, time
from urllib.parse import urlencode
from datetime import date, timedelta

# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


def clean_stores(store):
  #Since every store is in the same state, just set all state rows to the same value.
  store['state'] = 'FL'
  #Tallahassee is missing coords instead of dropping a store location
  store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]
  #drop any duplicate rows that may have the same latitude and longitude
  store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)
  return store

# Clean the stores DataFrame and initialize it
store = clean_stores(store)

#copied from lab
END = '2024-09-30'
START = '2024-09-01'
LANDFALL = pd.Timestamp('2024-09-26')

BASE = 'https://archive-api.open-meteo.com/v1/archive'



#Creating function including the necessary parameters
def fetch_store_weather(lat, lon, tz, start_date, end_date, tries=3):
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': tz,
    }
    #Error messages that are unacceptable and would cause errors
    RETRY_STATUSES = {429, 500, 502, 503, 504}
    #loop through the number of retry attempts
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
          #if above detects error, and below confirms this was our last try, show the error
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
          #if attempt is on the last try, then generate an error
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        #Goes through the daily fields for the weather of each store location
        store_weather = pd.DataFrame(r.json()['daily'])
        store_weather = store_weather.rename(columns={'time': 'date'})
        # Format weather date strings directly to standard 'YYYY-MM-DD'
        store_weather['date'] = pd.to_datetime(store_weather['date']).dt.strftime('%Y-%m-%d')
        store_weather['source'] = 'open-meteo'
        #returns a dataframe for the weather data of a specific store, one(1) store
        return store_weather

def get_all_weather(df_stores, start_date, end_date):
  #creates empty list for all individual store weather DataFrames for each store
    all_store_weather = []
    for i in range(len(df_stores)):
        store_id = df_stores.loc[i, 'store_id']
        city = df_stores.loc[i, 'city']
        lat = df_stores.loc[i, 'latitude']
        lon = df_stores.loc[i, 'longitude']
        # Timezone fixing, only pensacola is central, the rest are eastern
        tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'
        weather_df = fetch_store_weather(lat, lon, tz, start_date, end_date)
        if weather_df is not None:
            weather_df['store_id'] = store_id
            all_store_weather.append(weather_df)
    #Concatenate all individual store weather DataFrames into a single comprehensive master weather table
    master_weather = pd.concat(all_store_weather, ignore_index=True)
    # Ensure weather dates are perfectly formatted as localized standard datetime formats
    master_weather['date'] = pd.to_datetime(master_weather['date'])
    return master_weather

#Call the function to retrieve all weather data
df_weather = get_all_weather(store, START, END)

print(df_weather.head())

        date  precipitation_sum  windspeed_10m_max      source store_id
0 2024-09-01                6.4               18.7  open-meteo   FL-239
1 2024-09-02               22.0               12.7  open-meteo   FL-239
2 2024-09-03               32.7               20.6  open-meteo   FL-239
3 2024-09-04               21.2               12.6  open-meteo   FL-239
4 2024-09-05               20.8               15.2  open-meteo   FL-239


#Distribution center, one function

1. Generates weather data for distribution centers



In [2]:
def get_all_dc_weather(df_dc, start_date, end_date):
    #Coordinate and timezone mapping for the distribution center cities
    dc_geography = {
        'Jacksonville': {'lat': 30.3322, 'lon': -81.6557, 'tz': 'America/New_York'},
        'Atlanta': {'lat': 33.7490, 'lon': -84.3880, 'tz': 'America/New_York'},
        'Dallas': {'lat': 32.7767, 'lon': -96.7970, 'tz': 'America/Chicago'},
        'Miami': {'lat': 25.7617, 'lon': -80.1918, 'tz': 'America/New_York'},
        'Birmingham': {'lat': 33.5186, 'lon': -86.8104, 'tz': 'America/Chicago'}
    }

    all_dc_weather = []

    for i in range(len(df_dc)):
        dc_id = df_dc.loc[i, 'dc_id']
        city = df_dc.loc[i, 'city']

        #Retrieve data for each city as it loops
        location = dc_geography.get(city)
        #generates the weather data for each distribution center in the loop
        weather_df = fetch_store_weather(location['lat'], location['lon'], location['tz'], start_date, end_date)

        if weather_df is not None:
            weather_df['dc_id'] = dc_id
            all_dc_weather.append(weather_df)

    if all_dc_weather:
        return pd.concat(all_dc_weather, ignore_index=True)
    else:
        return pd.DataFrame()

#Generates weather data for the distribution centers
dc_weather = get_all_dc_weather(distribution, START, END)
display(dc_weather.head())

,date,precipitation_sum,windspeed_10m_max,source,dc_id
0,2024-09-01,1.3,19.6,open-meteo,DC-JAX
1,2024-09-02,5.4,11.3,open-meteo,DC-JAX
2,2024-09-03,26.5,26.0,open-meteo,DC-JAX
3,2024-09-04,17.2,22.5,open-meteo,DC-JAX
4,2024-09-05,6.6,15.4,open-meteo,DC-JAX


#hourly data for store weather instead of daily weather. 2 Functions
1. Generates hourly weather data for stores
2. Import timestamp function to fix transaction timedata in join function

In [18]:
def fetch_store_hourly_weather(lat, lon, tz, start_date, end_date, tries=3):
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,precipitation',
        'timezone': tz,
    }
    RETRY_STATUSES = {429, 500, 502, 503, 504}
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly

        # Parse the hourly fields using your reference format
        store_hourly_weather = pd.DataFrame(r.json()['hourly'])
        store_hourly_weather = store_hourly_weather.rename(columns={'time': 'datetime'})
        store_hourly_weather['datetime'] = pd.to_datetime(store_hourly_weather['datetime'])
        store_hourly_weather['source'] = 'open-meteo'
        return store_hourly_weather

def get_all_store_hourly_weather(df_stores, start_date, end_date):
    all_hourly_weather = []
    for i in range(len(df_stores)):
        store_id = df_stores.loc[i, 'store_id']
        city = df_stores.loc[i, 'city']
        lat = df_stores.loc[i, 'latitude']
        lon = df_stores.loc[i, 'longitude']
        tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'

        hourly_df = fetch_store_hourly_weather(lat, lon, tz, start_date, end_date)
        if hourly_df is not None:
            hourly_df['store_id'] = store_id
            all_hourly_weather.append(hourly_df)
            print(f"Successfully fetched hourly data for {store_id} ({city})")

    if all_hourly_weather:
        return pd.concat(all_hourly_weather, ignore_index=True)
    else:
        return pd.DataFrame()

# Fetch and combine hourly data for all stores
master_store_hourly_weather = get_all_store_hourly_weather(store, START, END)
display(master_store_hourly_weather.head())

Successfully fetched hourly data for FL-239 (Orlando)
Successfully fetched hourly data for FL-105 (Tampa)
Successfully fetched hourly data for FL-330 (Miami)
Successfully fetched hourly data for FL-412 (Jacksonville)


KeyboardInterrupt: 

In [16]:
# Handling the timestamps - creating a standardized date and timestamp format and keeping existing rows that don't have timestamps
# Using pandas standardized datetime
def format_timestamps(df):
    df = df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        errors="coerce"
    )

    return df

#1 function to join weather, transactions, and products

In [21]:
def perform_validated_joins(df_transactions, df_products, df_master_weather):
    #creates copies of data frames so that originals are not modified and ruined past this point for the joining.
    #We want the original data framed to be independent and clean, and not actually modified beyond their existing contents
    df_tx = df_transactions.copy()
    df_weather_local = df_master_weather.copy()

    #uses function to format timestamps
    df_tx = format_timestamps(df_tx)

    #create a standardized 'YYYY-MM-DD' date string from the formatted timestamp
    df_tx['date'] = df_tx['timestamp'].dt.strftime('%Y-%m-%d')

    #drop columns in transactions that overlap with the product catalog to avoid suffix duplication
    overlapping_cols = [col for col in ['product_name', 'category'] if col in df_tx.columns]
    if overlapping_cols:
        df_tx = df_tx.drop(columns=overlapping_cols)

    initial_row_count = len(df_tx)

    #merge the two dataframes using their skus
    df_tx_product = df_tx.merge(df_products, on='sku', how='left')

    #Now add weather data to the merge using ['store_id', 'date'] using a left join
    df_master_sales_weather = df_tx_product.merge(df_weather_local, on=['store_id', 'date'], how='left', indicator=True)

    #Use assertions to ensure all data is retained
    assert len(df_master_sales_weather) == initial_row_count, f'Rows join number went from {initial_row_count} to {len(df_master_sales_weather)}!'
    assert not (df_master_sales_weather['_merge'] == 'right_only').any(), 'rows that are not matched were found!'

    return df_master_sales_weather

#Asnwers Q3 questions using unclean product and transaction data

In [22]:
df_master_weather = get_all_weather(store, START, END)
# makes dates strings so that matching is accurate
df_master_weather['date'] = pd.to_datetime(df_master_weather['date']).dt.strftime('%Y-%m-%d')

#Actually run the join function
df_final_analysis = perform_validated_joins(transactions, product, df_master_weather)

#ensure all questions can be answered for Q3
print("Demonstration pipeline executed successfully!")
print(f"Row count before join: {len(transactions)}")
print(f"Row count after join: {len(df_final_analysis)}")
print(df_final_analysis[['transaction_id', 'store_id', 'date', 'product_name', 'precipitation_sum', 'windspeed_10m_max']].head())


Demonstration pipeline executed successfully!
Row count before join: 25562
Row count after join: 25562
  transaction_id store_id        date          product_name  \
0      TXN012636   FL-710  2024-09-13         Peanut Butter   
1      TXN000759   fl-412  2024-09-02           Canned Soup   
2      TXN003338   FL-105  2024-09-06  Toilet Paper 12-Roll   
3      TXN025020   FL-330  2024-09-30      Streaming Device   
4      TXN015797   FL-302  2024-09-15  Light Beer (24-pack)   

   precipitation_sum  windspeed_10m_max  
0                3.3               12.8  
1                NaN                NaN  
2                0.6               18.0  
3                1.3               16.2  
4                0.7               10.6  
